In [ ]:
import zipfile
import io
import os
import json
import pandas as pd

# 1. 대상 파일 리스트 (파일명이 정확한지 확인!) 88
zip_files = [
    '맨몸운동_Labeling.zip', 
    '맨몸운동_Labeling_new_220128.zip',
    '바벨_덤벨_Labeling.zip',
    '바벨_덤벨_Labeling_new_220128.zip'
]

# 2. 전수 조사 기반 진짜 번호 리스트
lunge_ids = [str(i) for i in range(113, 145)]
plank_ids = [str(i) for i in range(553, 561)]
pushup_ids = [str(i) for i in range(561, 593)]
squat_ids = ['15', '015', '16', '016'] 
all_target_ids = lunge_ids + plank_ids + pushup_ids + squat_ids

output_folder = 'extracted_exercises'
total_rows = []

# 폴더 초기화 (PermissionError 방지 로직 적용)
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
else:
    for item in os.listdir(output_folder):
        item_path = os.path.join(output_folder, item)
        # 폴더(.ipynb_checkpoints 등)가 아닌 파일만 삭제
        if os.path.isfile(item_path):
            os.remove(item_path)

print("🔍 데이터 통합 및 CSV 생성 시작 (1~2분 소요)...")

# [데이터 추출 및 통합 단계]
for root_zip_name in zip_files:
    if not os.path.exists(root_zip_name):
        print(f"⚠️ {root_zip_name} 파일이 폴더에 없어 건너뜁니다.")
        continue
        
    print(f"📦 {root_zip_name} 읽는 중...")
    with zipfile.ZipFile(root_zip_name, 'r') as root_zip:
        for inner_zip_name in root_zip.namelist():
            if inner_zip_name.endswith('.zip'):
                with root_zip.open(inner_zip_name) as inner_data:
                    zip_content = io.BytesIO(inner_data.read())
                    with zipfile.ZipFile(zip_content) as inner_zip:
                        for f in inner_zip.namelist():
                            if f.endswith('.json') and '3d' not in f:
                                # 파일명 끝에서 ID 추출
                                file_id = f.split('-')[-1].replace('.json', '')
                                if file_id in all_target_ids:
                                    try:
                                        # 바로 메모리에서 읽어서 total_rows에 추가
                                        data = json.loads(inner_zip.read(f))
                                        ex_name = data['type_info']['exercise']
                                        
                                        for i, frame in enumerate(data['frames']):
                                            # 여러 뷰 중 첫 번째 뷰 선택
                                            view_key = list(frame.keys())[0]
                                            pts = frame[view_key]['pts']
                                            
                                            row = {'filename': f, 'exercise': ex_name, 'frame_no': i}
                                            for part in ['Nose', 'Left Shoulder', 'Right Shoulder', 'Left Elbow', 'Right Elbow', 
                                                         'Left Hip', 'Right Hip', 'Left Knee', 'Right Knee', 'Left Ankle', 'Right Ankle']:
                                                if part in pts:
                                                    row[f'{part.lower().replace(" ", "_")}_x'] = pts[part]['x']
                                                    row[f'{part.lower().replace(" ", "_")}_y'] = pts[part]['y']
                                            total_rows.append(row)
                                    except: continue

# [결과 저장]
if total_rows:
    df = pd.DataFrame(total_rows)
    df.to_csv('final_all_exercises_data.csv', index=False, encoding='utf-8-sig')
    print("-" * 30)
    print("✅ 데이터 통합 성공!")
    print(f"📁 생성된 파일: final_all_exercises_data.csv (총 {len(df)}행)")
    print("\n📊 현재 확보된 운동 종목 및 개수:")
    print(df['exercise'].value_counts())
else:
    print("❌ 데이터를 찾지 못했습니다. 파일명이나 ID를 다시 확인해봐야 합니다.")

🔍 데이터 통합 및 CSV 생성 시작 (1~2분 소요)...
📦 맨몸운동_Labeling.zip 읽는 중...
📦 맨몸운동_Labeling_new_220128.zip 읽는 중...
📦 바벨_덤벨_Labeling.zip 읽는 중...
📦 바벨_덤벨_Labeling_new_220128.zip 읽는 중...
------------------------------
✅ 데이터 통합 성공!
📁 생성된 파일: final_all_exercises_data.csv (총 83002행)

📊 현재 확보된 운동 종목 및 개수:
exercise
스텝 백워드 다이나믹 런지    46526
푸시업               26624
플랭크                6624
스탠딩 사이드 크런치        3228
Name: count, dtype: int64


In [23]:
import zipfile
import io
import json

zip_files = ['바벨_덤벨_Labeling.zip', '바벨_덤벨_Labeling_new_220128.zip']
squat_finder = set()

print("🔎 스쿼트 번호를 역추적합니다...")

for zip_name in zip_files:
    with zipfile.ZipFile(zip_name, 'r') as root:
        for inner in root.namelist():
            if inner.endswith('.zip'):
                with root.open(inner) as i_data:
                    with zipfile.ZipFile(io.BytesIO(i_data.read())) as inner_zip:
                        for f in inner_zip.namelist():
                            if f.endswith('.json'):
                                # 우선 100개당 하나씩만 샘플링해서 이름을 확인 (속도 향상)
                                try:
                                    with inner_zip.open(f) as jf:
                                        data = json.load(jf)
                                        name = data['type_info']['exercise']
                                        if '스쿼트' in name:
                                            # '파일번호 : 운동이름' 출력
                                            num = f.split('-')[-1].replace('.json', '')
                                            squat_finder.add(f"{num} : {name}")
                                except: continue

print("-" * 30)
if squat_finder:
    print("✅ 찾았습니다! 스쿼트 관련 번호들:")
    for res in squat_finder: print(res)
else:
    print("❌ 바벨 파일에도 스쿼트가 없네요. '맨몸운동' zip 파일을 더 다운로드 받아야 할 것 같습니다.")

🔎 스쿼트 번호를 역추적합니다...
------------------------------
✅ 찾았습니다! 스쿼트 관련 번호들:
317 : 바벨 스쿼트
321 : 바벨 스쿼트
319 : 바벨 스쿼트
316 : 바벨 스쿼트
323 : 바벨 스쿼트
320 : 바벨 스쿼트
325 : 바벨 스쿼트
327 : 바벨 스쿼트
322 : 바벨 스쿼트
318 : 바벨 스쿼트
315 : 바벨 스쿼트
314 : 바벨 스쿼트
326 : 바벨 스쿼트
328 : 바벨 스쿼트
324 : 바벨 스쿼트
313 : 바벨 스쿼트


In [25]:
import zipfile
import io
import json
import os

# 조사할 파일 리스트
zip_files = ['맨몸운동_Labeling.zip', '맨몸운동_Labeling_new_220128.zip']
bodyweight_squat_finder = set()

print("🔎 맨몸운동 파일 내에서 '스쿼트'를 다시 추적합니다...")

for root_zip_name in zip_files:
    if not os.path.exists(root_zip_name):
        print(f"⚠️ {root_zip_name} 파일이 없습니다.")
        continue
        
    with zipfile.ZipFile(root_zip_name, 'r') as root_zip:
        inner_zip_names = [f for f in root_zip.namelist() if f.endswith('.zip')]
        
        for inner_name in inner_zip_names:
            # 에러 방지: 데이터를 먼저 완전히 읽어온 뒤 BytesIO로 변환
            inner_zip_data = root_zip.read(inner_name)
            with zipfile.ZipFile(io.BytesIO(inner_zip_data)) as inner_zip:
                # JSON 파일들만 골라내기
                json_files = [f for f in inner_zip.namelist() if f.endswith('.json') and '3d' not in f]
                
                # 속도를 위해 각 내부 zip마다 앞부분 20개 정도만 샘플로 확인해봅니다.
                for f in json_files[:20]: 
                    try:
                        with inner_zip.open(f) as jf:
                            data = json.loads(jf.read().decode('utf-8'))
                            ex_name = data.get('type_info', {}).get('exercise', '')
                            if '스쿼트' in ex_name:
                                # 파일 번호(ID) 추출
                                num = f.split('-')[-1].replace('.json', '')
                                bodyweight_squat_finder.add(f"{num} : {ex_name}")
                    except:
                        continue

print("-" * 30)
if bodyweight_squat_finder:
    print("✅ 찾았습니다! 맨몸운동 속 스쿼트 번호들:")
    for res in sorted(list(bodyweight_squat_finder)):
        print(res)
else:
    print("❌ 맨몸운동 파일에는 '스쿼트'가 정말 없네요.")
    print("팀 프로젝트용 스쿼트 데이터는 아까 찾은 '바벨 스쿼트(313~328)' 번호를 사용하면 됩니다!")

🔎 맨몸운동 파일 내에서 '스쿼트'를 다시 추적합니다...
------------------------------
❌ 맨몸운동 파일에는 '스쿼트'가 정말 없네요.
팀 프로젝트용 스쿼트 데이터는 아까 찾은 '바벨 스쿼트(313~328)' 번호를 사용하면 됩니다!


In [26]:
import zipfile
import io
import os
import json
import pandas as pd

# 1. 모든 압축 파일 리스트
zip_files = [
    '맨몸운동_Labeling.zip', 
    '맨몸운동_Labeling_new_220128.zip',
    '바벨_덤벨_Labeling.zip',
    '바벨_덤벨_Labeling_new_220128.zip'
]

# 2. 확정된 종목별 ID 리스트
lunge_ids = [str(i) for i in range(113, 145)]  # 런지
plank_ids = [str(i) for i in range(553, 561)]  # 플랭크
pushup_ids = [str(i) for i in range(561, 593)] # 푸시업
squat_ids = [str(i) for i in range(313, 329)]  # 스쿼트 (바벨 스쿼트)
all_target_ids = lunge_ids + plank_ids + pushup_ids + squat_ids

total_rows = []

print("🚀 4종 운동 데이터 통합 추출을 시작합니다 (약 1~2분 소요)...")

for root_zip_name in zip_files:
    if not os.path.exists(root_zip_name):
        print(f"⚠️ {root_zip_name} 파일이 없어 건너뜁니다.")
        continue
        
    print(f"📦 {root_zip_name} 읽는 중...")
    with zipfile.ZipFile(root_zip_name, 'r') as root_zip:
        inner_zip_names = [f for f in root_zip.namelist() if f.endswith('.zip')]
        
        for inner_name in inner_zip_names:
            inner_zip_data = root_zip.read(inner_name)
            with zipfile.ZipFile(io.BytesIO(inner_zip_data)) as inner_zip:
                for f in inner_zip.namelist():
                    # 3D 파일 제외, JSON 파일만 대상
                    if f.endswith('.json') and '3d' not in f:
                        file_id = f.split('-')[-1].replace('.json', '')
                        if file_id in all_target_ids:
                            try:
                                data = json.loads(inner_zip.read(f).decode('utf-8'))
                                ex_name = data['type_info']['exercise']
                                
                                for i, frame in enumerate(data['frames']):
                                    view_key = list(frame.keys())[0]
                                    pts = frame[view_key]['pts']
                                    
                                    row = {'filename': f, 'exercise': ex_name, 'frame_no': i}
                                    # 주요 관절 좌표 추출
                                    for part in ['Nose', 'Left Shoulder', 'Right Shoulder', 'Left Elbow', 'Right Elbow', 
                                                 'Left Hip', 'Right Hip', 'Left Knee', 'Right Knee', 'Left Ankle', 'Right Ankle']:
                                        if part in pts:
                                            row[f'{part.lower().replace(" ", "_")}_x'] = pts[part]['x']
                                            row[f'{part.lower().replace(" ", "_")}_y'] = pts[part]['y']
                                    total_rows.append(row)
                            except: continue

# 3. CSV 저장 및 결과 출력
if total_rows:
    df = pd.DataFrame(total_rows)
    df.to_csv('final_4_exercises_dataset.csv', index=False, encoding='utf-8-sig')
    print("-" * 30)
    print("✨ 데이터 통합 완료!")
    print(f"📁 파일명: final_4_exercises_dataset.csv (총 {len(df)}행)")
    print("\n📊 최종 종목별 분포:")
    print(df['exercise'].value_counts())
else:
    print("❌ 데이터를 찾지 못했습니다. 파일명을 다시 확인해주세요.")

🚀 4종 운동 데이터 통합 추출을 시작합니다 (약 1~2분 소요)...
📦 맨몸운동_Labeling.zip 읽는 중...
📦 맨몸운동_Labeling_new_220128.zip 읽는 중...
📦 바벨_덤벨_Labeling.zip 읽는 중...
📦 바벨_덤벨_Labeling_new_220128.zip 읽는 중...
------------------------------
✨ 데이터 통합 완료!
📁 파일명: final_4_exercises_dataset.csv (총 102752행)

📊 최종 종목별 분포:
exercise
스텝 백워드 다이나믹 런지    46526
푸시업               26624
바벨 스쿼트            22978
플랭크                6624
Name: count, dtype: int64
